# Preparação dos dados para análise

## Inicialização

In [2]:
# carregando as bibliotecas necessárias
import pandas as pd
import numpy as np

## Carregando os dados

In [3]:
# carregando o dataset
df = pd.read_csv('datasets/Churn.csv')

## Avaliação incial

In [4]:
# avaliação inicial do dataset
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           9091 non-null   float64
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(3), int64(8), str(3)
memory usage: 1.1 MB


In [5]:
# verificando uma amostra dos dados
print(df.sample(5))

      RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
2829       2830    15759913  Trentini          553   Germany    Male   43   
7419       7420    15622507  Hamilton          748   Germany  Female   40   
9816       9817    15649508      Chin          643     Spain    Male   48   
239         240    15795149   Stevens          703    France    Male   28   
1321       1322    15704483   Lorenzo          724    France    Male   40   

      Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
2829     6.0   85200.82              2          1               1   
7419     3.0  103499.09              2          0               0   
9816     8.0       0.00              2          1               0   
239      2.0   81173.83              2          0               1   
1321     6.0       0.00              2          0               0   

      EstimatedSalary  Exited  
2829        160574.09       0  
7419         38153.19       0  
9816        174729.30     

In [6]:
# avaliando a estatística descritiva inicial do dataset
print(df.describe())

         RowNumber    CustomerId   CreditScore           Age       Tenure  \
count  10000.00000  1.000000e+04  10000.000000  10000.000000  9091.000000   
mean    5000.50000  1.569094e+07    650.528800     38.921800     4.997690   
std     2886.89568  7.193619e+04     96.653299     10.487806     2.894723   
min        1.00000  1.556570e+07    350.000000     18.000000     0.000000   
25%     2500.75000  1.562853e+07    584.000000     32.000000     2.000000   
50%     5000.50000  1.569074e+07    652.000000     37.000000     5.000000   
75%     7500.25000  1.575323e+07    718.000000     44.000000     7.000000   
max    10000.00000  1.581569e+07    850.000000     92.000000    10.000000   

             Balance  NumOfProducts    HasCrCard  IsActiveMember  \
count   10000.000000   10000.000000  10000.00000    10000.000000   
mean    76485.889288       1.530200      0.70550        0.515100   
std     62397.405202       0.581654      0.45584        0.499797   
min         0.000000       1.00000

**Resumo da avaliação inicial do dataset**

O conjunto de dados consiste de 10 mil linhas e 14 colunas. Os dados são de fácil entendimento. É necessário avaliar com o cliente se a coluna 'RowNumber' tem alguma relevancia para ele, para este estudo ela será removida para reduzir o tamanho do arquivo.

**Pré-processamento necessário**

- remover a coluna 'RowNumber'
- ajustar os nomes das colunas: aplicar procedimento de limpeza e ajuste padrão (tudo em minúsculo, limpar espaços em branco que houver) e separar palavras por '_'.

**Preparação dos dados**

- avaliar valores ausêntes na coluna 'Tenure'
- avaliar tipo de dado da coluna 'Tenure'
- avaliar linhas completamente duplicadas
- avaliar clientes duplicados na coluna 'CustomerId'


**Melhoramentos iniciais nos dados**

- transformar os dados da coluna 'geography' em categorias numéricas
- transformar os dados da coluna 'gender' em 0 para Female e 1 para Male

## Pré-processamento

### Removendo coluna 'RowNumber'

In [7]:
# removendo a coluna RowNumber do dataset
df = df.drop(['RowNumber'], axis=1)

In [8]:
# confirmando a remoção da coluna RowNumber
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CustomerId       10000 non-null  int64  
 1   Surname          10000 non-null  str    
 2   CreditScore      10000 non-null  int64  
 3   Geography        10000 non-null  str    
 4   Gender           10000 non-null  str    
 5   Age              10000 non-null  int64  
 6   Tenure           9091 non-null   float64
 7   Balance          10000 non-null  float64
 8   NumOfProducts    10000 non-null  int64  
 9   HasCrCard        10000 non-null  int64  
 10  IsActiveMember   10000 non-null  int64  
 11  EstimatedSalary  10000 non-null  float64
 12  Exited           10000 non-null  int64  
dtypes: float64(3), int64(7), str(3)
memory usage: 1015.8 KB
None


### Ajuste do cabeçalho

In [9]:
# verificando os títulos atuais das colunas do dataset
print(df.columns)

Index(['CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age',
       'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember',
       'EstimatedSalary', 'Exited'],
      dtype='str')


In [11]:
# ajustar os títulos das colunas com protocolo padrão
new_col_names = []
for old_name in df.columns:
    name_stripped = old_name.strip()
    name_lowered = name_stripped.lower()
    name_no_spaces = name_lowered.replace(' ', '_')
    new_col_names.append(name_no_spaces)

df.columns = new_col_names

print(df.columns)

Index(['customerid', 'surname', 'creditscore', 'geography', 'gender', 'age',
       'tenure', 'balance', 'numofproducts', 'hascrcard', 'isactivemember',
       'estimatedsalary', 'exited'],
      dtype='str')


In [12]:
# renomeando as colunas com o padrão snake_case
df = df.rename(columns={
    'customerid': 'customer_id',
    'creditscore': 'credit_score',
    'numofproducts': 'num_of_products',
    'hascrcard': 'has_cr_card',
    'isactivemember': 'is_active_member',
    'estimatedsalary': 'estimated_salary'
})

print(df.columns)

Index(['customer_id', 'surname', 'credit_score', 'geography', 'gender', 'age',
       'tenure', 'balance', 'num_of_products', 'has_cr_card',
       'is_active_member', 'estimated_salary', 'exited'],
      dtype='str')
